In [1]:
# Experiment 5: Token economics - cost and rate-limit budget for a feature BEFORE building it
# Feature: "Ask the course assistant" for 1,000 daily users
from google.colab import userdata
from google import genai
import pandas as pd

MODEL = "gemini-3.1-flash-lite"
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"), http_options={"timeout": 30000})

# 1) Measure real token counts on sample requests (system prompt + retrieved context + question)
SYSTEM = "You are a course assistant for Generative AI. Answer briefly and cite the course handout."
CONTEXT = "Module III covers embeddings, chunking, MongoDB, Chroma and RAG evaluation with Hit Rate@k and MRR. " * 8
questions = ["What is RAG?", "Explain MRR with an example.", "రాగ్ అంటే ఏమిటి?", "आरएजी क्या है?"]
rows = []
for q in questions:
    r = client.models.generate_content(model=MODEL, contents=SYSTEM + "\n" + CONTEXT + "\nQuestion: " + q)
    rows.append({"question": q, "input_tokens": r.usage_metadata.prompt_token_count,
                 "output_tokens": r.usage_metadata.candidates_token_count})
sample = pd.DataFrame(rows); print(sample.to_string(index=False))

# 2) Budget assumptions (edit these)
USERS_PER_DAY = 1000
QUESTIONS_PER_USER = 5
PEAK_HOUR_SHARE = 0.20                       # 20% of daily traffic in the busiest hour
PRICE_IN, PRICE_OUT = 0.10, 0.40             # illustrative USD per 1M tokens - check the current pricing page
RPM_LIMIT, RPD_LIMIT = 15, 1000              # illustrative free-tier limits - check your AI Studio dashboard
CONTEXT_LIMIT = 1_000_000                     # model context window (tokens)

avg_in, avg_out = sample.input_tokens.mean(), sample.output_tokens.mean()
req_day = USERS_PER_DAY * QUESTIONS_PER_USER
cost_req = (avg_in * PRICE_IN + avg_out * PRICE_OUT) / 1e6
peak_rpm = req_day * PEAK_HOUR_SHARE / 60

budget = {
    "avg input tokens / request": round(avg_in), "avg output tokens / request": round(avg_out),
    "requests / day": req_day, "tokens / day": round(req_day * (avg_in + avg_out)),
    "cost / request (USD)": round(cost_req, 6), "cost / day (USD)": round(cost_req * req_day, 2),
    "cost / month (USD)": round(cost_req * req_day * 30, 2), "cost / month (INR @ 84)": round(cost_req * req_day * 30 * 84),
    "peak requests / minute": round(peak_rpm, 1),
    "fits free-tier RPM?": peak_rpm <= RPM_LIMIT, "fits free-tier RPD?": req_day <= RPD_LIMIT,
    "context used per request": f"{avg_in / CONTEXT_LIMIT:.4%}",
}
print("\nCost budget note:")
for k, v in budget.items(): print(f"  {k:28s}: {v}")
print("\nDecision: if RPM/RPD do not fit, add caching, a paid tier, or a queue before launch.")

                    question  input_tokens  output_tokens
                What is RAG?           221             56
Explain MRR with an example.           223            139
            రాగ్ అంటే ఏమిటి?           222             77
              आरएजी क्या है?           222             65

Cost budget note:
  avg input tokens / request  : 222
  avg output tokens / request : 84
  requests / day              : 5000
  tokens / day                : 1531250
  cost / request (USD)        : 5.6e-05
  cost / day (USD)            : 0.28
  cost / month (USD)          : 8.39
  cost / month (INR @ 84)     : 704
  peak requests / minute      : 16.7
  fits free-tier RPM?         : False
  fits free-tier RPD?         : False
  context used per request    : 0.0222%

Decision: if RPM/RPD do not fit, add caching, a paid tier, or a queue before launch.
